This page showcases a few examples using open data. Underneath the hood, this is a jupyter notebook that you can download and replicate, including modifying it for other data.

In [4]:
import pandas as pd
import numpy as np

# Close up of the Dominick's open scanner dataset

Throughout the notebook, we will focus on scanner data from the Dominick's Finer Food grocery retailer that used to be in business in Chicago, USA in the 90s. The University of Chicago Booth School of Business makes this data available for research purposes.[^1]

## Overview of the data

The data comes in 4 separate parts:

-   A definition of weeks
-   A definition of stores
-   The unique product (i.e. UPC) files per category
-   The sales (i.e. movement) files, again per category. This is total turnover and quantities sold per UPC per week.

As NSOs typically get a week's file that will include all this information (and things like week definitions would typically be defined as time goes on), we should join these 4 datasets into one. NSOs typically also receive all the data in one batch (irrespective of the category of the products, indeed the assigning of product the right category is classification). Thus for demonstration purposes, we'll join two product categories (movements and UPC files) into one big file.

## Visual overview of the data flow

Thus we can look at the raw flow to see how this works

![Example data flow](../content/images/dominicks-flow.drawio.svg)


[^1]: Dataset documented in detail on the [Price Statistics Open Data Catalogue](https://un-task-team-for-scanner-data.github.io/price-stats-data-catalogue/). See also the [Dataset home page in the Booth school of business website](https://www.chicagobooth.edu/research/kilts/research-data/dominicks).

# Data Standardization examples

As there are 4 different files that have to be queried and combined to work with each category, we can focus on some data standardization examples on two of the specific files - the weekly data and the product specific data

## Weekly data

The raw weekly data comes unstructured. Indeed it doesn't even have a header, so we have to specify it manually when we import the data

In [10]:
#| echo: true
#| code-fold: true

weeks_url = "https://raw.githubusercontent.com/eurostat/dff/master/CSV/weeks.csv"
df = pd.read_csv(
    weeks_url, 
    header=None, 
    names=['WEEK','START','END','SPECIAL_EVENTS'])
df.head()


,WEEK,START,END,SPECIAL_EVENTS
0,1,09/14/89,09/20/89,NaN
1,2,09/21/89,09/27/89,NaN
2,3,09/28/89,10/04/89,NaN
3,4,10/05/89,10/11/89,NaN
4,5,10/12/89,10/18/89,NaN


Investigating the variable types, we see that we will have to process them to work with them.

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   WEEK            400 non-null    int64 
 1   START           400 non-null    object
 2   END             400 non-null    object
 3   SPECIAL_EVENTS  69 non-null     object
dtypes: int64(1), object(3)
memory usage: 12.6+ KB


To work with the data, we can do a few things:
-   convert the dates into actual datetime objects to make manipulation easy
-   create a custom variable that tells us what reference period the weekly file belongs to (say `"1996-01"`) as we measure the CPI on a monthly basis
-   evaluate if a week falls cleanly within the month (does it start at the end of one month and end in the beginning of the next one) and if it does fall within the month, what week number is it?[^1]

[^1]: We need to assign prices/quanitities to the months they happen in, however if we receive data that stradles two months, there may not be a clean way to separate it out. We we can start with the first *clean* week. Furthermore, as we have a deadline to publish the CPI (typically early in the month following the one we are measuring), it is sometimes challenging to include all the weekly data. See [Chapter 3 of the Eurostat Multilateral guide](https://ec.europa.eu/eurostat/fr/web/products-manuals-and-guidelines/-/KS-GQ-21-020) for more on aggregation across time. 

In [ ]:
#| echo: true
#| code-fold: true

# First off, cast the START and END columns to datetime
df['START'] = pd.to_datetime(df['START'], format='%m/%d/%y')
df['END'] = pd.to_datetime(df['END'], format='%m/%d/%y')

# Now we can extract the reference period (month) and determine if the week is fully in the month
df['REF_PERIOD'] = df['START'].dt.strftime('%Y-%m')

# Now we can determine if the week is fully in the month by checking if the month of the start and end dates are the same
df['WEEK_FULLY_IN_MONTH'] = np.where(
        df['START'].dt.month == df['END'].dt.month, 
        True, 
        np.nan
    )

valid_weeks = df['WEEK_FULLY_IN_MONTH'].notna()
df['WEEK_OF_MONTH'] = (
        valid_weeks
        .groupby(df['REF_PERIOD'])
        .cumsum()
        .where(valid_weeks, np.nan)
    )
df.head(10)

,WEEK,START,END,SPECIAL_EVENTS,REF_PERIOD,WEEK_FULLY_IN_MONTH,WEEK_OF_MONTH
0,1,1989-09-14,1989-09-20,NaN,1989-09,1.0,1.0
1,2,1989-09-21,1989-09-27,NaN,1989-09,1.0,2.0
2,3,1989-09-28,1989-10-04,NaN,1989-09,NaN,NaN
3,4,1989-10-05,1989-10-11,NaN,1989-10,1.0,1.0
4,5,1989-10-12,1989-10-18,NaN,1989-10,1.0,2.0
5,6,1989-10-19,1989-10-25,NaN,1989-10,1.0,3.0
6,7,1989-10-26,1989-11-01,Halloween,1989-10,NaN,NaN
7,8,1989-11-02,1989-11-08,NaN,1989-11,1.0,1.0
8,9,1989-11-09,1989-11-15,NaN,1989-11,1.0,2.0
9,10,1989-11-16,1989-11-22,NaN,1989-11,1.0,3.0


# Classification

The two categories we picked could be assinged to two COICOP 2018 classes:[^2]

-   Laundry Detergents --> could be part of COICOP 13.1.2.0 - Other appliances, articles and products for personal care (ND)
-   Grooming Products --> could be part of COCIOP 05.6.1.1 - Household cleaning and maintenance products (ND)

Note - if we had a more extensive example, other Dominick's categories such as Fabric Softeners, Toothbrushes, and others may fall into one of these categories. 

[^2]: See [COICOP 2018 in full (as hosted on the New Zealand Aria database)](https://aria.stats.govt.nz/aria/#ClassificationView:uri=http://stats.govt.nz/cms/ClassificationVersion/hTe8vaVj73ScFSJe)

# Data filtering examples

To showcase what would happen if there are data issues -- let us consider the example that Kevin Fox raised during the 2026 Ottawa Group - see his presentation on the "[Curious case of the exploding GEKS index](https://ottawagroup2026.stat.gov.pl/en/DownloadFile?nrSesji=2&nrWystapienia=1&nrPliku=2)". 

Specifically, Kevin used the case of Grooming products - 